In [1]:
# CELL 1 — INSTALL
!pip install transformers torch Pillow tqdm hnswlib -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# CELL 2 — IMPORTS
import os
import numpy as np
import pandas as pd
import torch
import hnswlib
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import shutil

In [3]:
# CELL 3 — CONFIG
ALPHA      = 0.5
TOP_K      = 15   # retrieve top 15, slice at 5/10/15 after ITM

# Inputs
EMB_DIR      = "/kaggle/input/datasets/ankithkini/embeddings"
CAPTIONS_CSV = "/kaggle/input/datasets/ankithkini/preprocessed-deepfashion-input/captions_new.csv"
QUERY_DIR    = "/kaggle/input/datasets/ankithkini/preprocessed-deepfashion-input/split_images/query"
GALLERY_DIR  = "/kaggle/input/datasets/ankithkini/preprocessed-deepfashion-input/split_images/gallery"

# Embedding paths
GAL_VIS_EMB  = os.path.join(EMB_DIR, "visual",  "gallery", "embeddings.npy")
GAL_VIS_FN   = os.path.join(EMB_DIR, "visual",  "gallery", "filenames.npy")
GAL_VIS_IDS  = os.path.join(EMB_DIR, "visual",  "gallery", "item_ids.npy")
GAL_CAP_EMB  = os.path.join(EMB_DIR, "caption", "gallery", "embeddings.npy")
GAL_CAP_FN   = os.path.join(EMB_DIR, "caption", "gallery", "filenames.npy")
QRY_VIS_EMB  = os.path.join(EMB_DIR, "visual",  "query",   "embeddings.npy")
QRY_VIS_FN   = os.path.join(EMB_DIR, "visual",  "query",   "filenames.npy")
QRY_VIS_IDS  = os.path.join(EMB_DIR, "visual",  "query",   "item_ids.npy")

OUTPUT_DIR   = "/kaggle/working/results_alpha_0_5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Alpha: {ALPHA}")
for p in [GAL_VIS_EMB, GAL_CAP_EMB, QRY_VIS_EMB, CAPTIONS_CSV]:
    print(f"  exists={os.path.exists(p)}  {p}")

Alpha: 0.5
  exists=True  /kaggle/input/datasets/ankithkini/embeddings/visual/gallery/embeddings.npy
  exists=True  /kaggle/input/datasets/ankithkini/embeddings/caption/gallery/embeddings.npy
  exists=True  /kaggle/input/datasets/ankithkini/embeddings/visual/query/embeddings.npy
  exists=True  /kaggle/input/datasets/ankithkini/preprocessed-deepfashion-input/captions_new.csv


In [4]:
# CELL 4 — LOAD EMBEDDINGS
gal_vis_emb = np.load(GAL_VIS_EMB)
gal_vis_fn  = np.load(GAL_VIS_FN)
gal_vis_ids = np.load(GAL_VIS_IDS)
gal_cap_emb = np.load(GAL_CAP_EMB)
gal_cap_fn  = np.load(GAL_CAP_FN)
qry_vis_emb = np.load(QRY_VIS_EMB)
qry_vis_fn  = np.load(QRY_VIS_FN)
qry_vis_ids = np.load(QRY_VIS_IDS)

print(f"Gallery visual  : {gal_vis_emb.shape}")
print(f"Gallery caption : {gal_cap_emb.shape}")
print(f"Query visual    : {qry_vis_emb.shape}")

# Verify gallery visual and caption are aligned
assert (gal_vis_fn == gal_cap_fn).all(), "Gallery visual/caption filename mismatch!"
print("Gallery alignment OK")

Gallery visual  : (12612, 512)
Gallery caption : (12612, 512)
Query visual    : (14218, 512)
Gallery alignment OK


In [5]:
# CELL 5 — FUSE GALLERY EMBEDDINGS
# vi = alpha * phi_V + (1 - alpha) * phi_T, then L2 normalize
gal_fused = ALPHA * gal_vis_emb + (1 - ALPHA) * gal_cap_emb
gal_fused = gal_fused / np.linalg.norm(gal_fused, axis=1, keepdims=True)

# Query uses visual only per spec
qry_fused = qry_vis_emb  # already L2 normalized

print(f"Fused gallery shape : {gal_fused.shape}")
print(f"Sample norm (should be ~1.0): {np.linalg.norm(gal_fused[:3], axis=1)}")

Fused gallery shape : (12612, 512)
Sample norm (should be ~1.0): [0.99999994 0.99999994 0.99999994]


In [6]:
# CELL 6 — BUILD HNSW INDEX
DIM = gal_fused.shape[1]

index = hnswlib.Index(space="cosine", dim=DIM)
index.init_index(max_elements=len(gal_fused), ef_construction=200, M=16)
index.add_items(gal_fused, np.arange(len(gal_fused)))
index.set_ef(50)

print(f"HNSW index built with {index.get_current_count()} vectors")

HNSW index built with 12612 vectors


In [7]:
# CELL 7 — LOAD BLIP v1 FOR ITM
from transformers import BlipProcessor, BlipForImageTextRetrieval

device       = "cuda" if torch.cuda.is_available() else "cpu"
ITM_MODEL_ID = "Salesforce/blip-itm-base-coco"

itm_processor = BlipProcessor.from_pretrained(ITM_MODEL_ID)
itm_model     = BlipForImageTextRetrieval.from_pretrained(
    ITM_MODEL_ID,
    torch_dtype=torch.float16
).to(device)
itm_model.eval()
print(f"BLIP ITM loaded on {device}")

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/895M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/895M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

BlipForImageTextRetrieval LOAD REPORT from: Salesforce/blip-itm-base-coco
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_encoder.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BLIP ITM loaded on cuda


In [8]:
# CELL 8 — LOAD CAPTIONS LOOKUP
captions_df = pd.read_csv(CAPTIONS_CSV)
captions_df["filename"] = captions_df["cropped_image_path"].apply(os.path.basename)
caption_lookup = dict(zip(captions_df["filename"], captions_df["caption"]))
print(f"Caption lookup: {len(caption_lookup)} entries")

Caption lookup: 52712 entries


In [9]:
# CELL 9 — ITM SCORING FUNCTION
import torch.nn.functional as F

def itm_score(query_image_path, candidate_captions):
    try:
        query_image = Image.open(query_image_path).convert("RGB")
    except Exception:
        return [0.0] * len(candidate_captions)

    cleaned = [c if isinstance(c, str) and c.strip() else "a clothing item"
               for c in candidate_captions]

    inputs = itm_processor(
        images=[query_image] * len(cleaned),
        text=cleaned,
        return_tensors="pt",
        padding=True
    ).to(device, torch.float16)

    with torch.no_grad():
        itm_out = itm_model(**inputs)[0]
        scores  = F.softmax(itm_out, dim=-1)[:, 1].tolist()

    return scores

In [10]:
# CELL 10 — RETRIEVAL + ITM RERANKING
results = []

for i, (qry_emb, qry_fn, qry_id) in enumerate(tqdm(
    zip(qry_fused, qry_vis_fn, qry_vis_ids),
    total=len(qry_fused),
    desc="Retrieval+ITM"
)):
    # ANN retrieval — top 15
    labels, distances = index.knn_query(qry_emb.reshape(1, -1), k=TOP_K)
    top_indices       = labels[0]
    top_fns           = gal_vis_fn[top_indices]
    top_ids           = gal_vis_ids[top_indices]
    top_captions      = [caption_lookup.get(fn, "a clothing item") for fn in top_fns]

    query_img_path = os.path.join(QUERY_DIR, qry_fn)

    # Single batched ITM pass for all 15
    itm_scores_all = itm_score(query_img_path, top_captions)

    reranked_5  = top_ids[:5][np.argsort(itm_scores_all[:5])[::-1]]
    reranked_10 = top_ids[:10][np.argsort(itm_scores_all[:10])[::-1]]
    reranked_15 = top_ids[:15][np.argsort(itm_scores_all[:15])[::-1]]

    results.append({
        "query_filename" : qry_fn,
        "query_item_id"  : qry_id,
        "reranked_5"     : reranked_5.tolist(),
        "reranked_10"    : reranked_10.tolist(),
        "reranked_15"    : reranked_15.tolist()
    })

print(f"Retrieval complete for {len(results)} queries.")


Retrieval+ITM:  62%|██████▏   | 8847/14218 [41:42<25:30,  3.51it/s]The channel dimension is ambiguous. Got image shape torch.Size([3, 120, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 120, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 120, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_forma

Retrieval complete for 14218 queries.


In [11]:
# CELL 11 — COMPUTE METRICS
def recall_at_k(results, k):
    key = f"reranked_{k}"
    hits = sum(1 for r in results if r["query_item_id"] in r[key])
    return hits / len(results)

def ndcg_at_k(results, k):
    key = f"reranked_{k}"
    scores = []
    for r in results:
        retrieved = r[key]
        dcg  = sum(1.0 / np.log2(rank + 2) for rank, rid in enumerate(retrieved) if rid == r["query_item_id"])
        idcg = 1.0 / np.log2(2)
        scores.append(dcg / idcg)
    return np.mean(scores)

def map_at_k(results, k):
    key = f"reranked_{k}"
    aps = []
    for r in results:
        retrieved = r[key]
        hits, precision_sum = 0, 0.0
        for rank, rid in enumerate(retrieved):
            if rid == r["query_item_id"]:
                hits += 1
                precision_sum += hits / (rank + 1)
        aps.append(precision_sum / hits if hits > 0 else 0.0)
    return np.mean(aps)

print(f"{'Metric':<12} {'K=5':>8} {'K=10':>8} {'K=15':>8}")
print("-" * 40)
for metric_name, fn in [("Recall", recall_at_k), ("NDCG", ndcg_at_k), ("mAP", map_at_k)]:
    vals = [fn(results, k) for k in [5, 10, 15]]
    print(f"{metric_name:<12} {vals[0]:>8.4f} {vals[1]:>8.4f} {vals[2]:>8.4f}")

Metric            K=5     K=10     K=15
----------------------------------------
Recall         0.3745   0.4519   0.4946
NDCG           0.3404   0.3832   0.4029
mAP            0.2387   0.2278   0.2136


In [12]:
# CELL 12 — SAVE RESULTS
rows = []
for r in results:
    for k in [5, 10, 15]:
        rows.append({
            "query_filename" : r["query_filename"],
            "query_item_id"  : r["query_item_id"],
            "K"              : k,
            "Recall"         : recall_at_k([r], k),
            "NDCG"           : ndcg_at_k([r], k),
            "mAP"            : map_at_k([r], k)
        })

results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(OUTPUT_DIR, "metrics_alpha_0_5.csv"), index=False)
print(f"Saved to {OUTPUT_DIR}/metrics_alpha_0_5.csv")

shutil.make_archive(f"/kaggle/working/results_alpha_0_5", "zip", OUTPUT_DIR)
print(f"ZIP saved.")

Saved to /kaggle/working/results_alpha_0_5/metrics_alpha_0_5.csv
ZIP saved.
